# BAN 8101 Predictive Analytics
## Assignment 1 — Synthetic Customer Dataset Generator

### Purpose

This notebook generates the individualized synthetic dataset that you will use for Assignment 1.

Each student must use their **own Student ID number as the random seed**.

Because your Student ID is used as the seed:

- your dataset will be reproducible,
- your results may differ from your classmates' results,
- and you should not replace the seed with another number.

### Important

Replace the placeholder Student ID below with your own Student ID.

If you need additional random values anywhere in Assignment 1, use the same seeded random number generator (`rng`) created in this notebook.

# 2. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd

print("Libraries loaded successfully.")

Libraries loaded successfully.


# 1. Enter Your Student ID

Replace the example value below with your own Student ID.

Example:

```python
STUDENT_ID = 123456789
```

Do not use a classmate's Student ID.

Do not create a new random seed elsewhere in the assignment unless specifically instructed.

In [2]:
# Replace this example with your own Student ID number
STUDENT_ID = 123456789

# Create the random number generator
rng = np.random.default_rng(STUDENT_ID)

print("Student ID seed:", STUDENT_ID)

Student ID seed: 123456789


# 3. Dataset Design

This dataset represents customers of a subscription-based company.

It contains predictor variables such as:

- customer tenure
- monthly charges
- support activity
- service usage
- contract type
- automatic payment status
- age
- discount rate
- customer satisfaction

It also contains two target variables:

### Regression Target

`annual_spending`

This is a continuous numerical variable and will be used for your linear regression, Ridge, Lasso, cross-validation, and feature-engineering work.

### Classification Target

`churn`

This is a binary target:

- `0 = customer stays`
- `1 = customer churns`

It will be used for your logistic regression and classification evaluation work.

# 4. Generate the Predictor Variables

The code below creates 2,000 synthetic customers.

You should **run the code as provided**.

Do not manually alter individual customer records.

In [3]:
n = 2000

# Customer tenure in months: 1 to 72 months
tenure_months = rng.integers(1, 73, n)

# Monthly charge: centered around $75, limited to a realistic range
monthly_charge = np.clip(
    rng.normal(75, 20, n),
    20,
    150
)

# Recent number of support tickets
support_tickets = np.clip(
    rng.poisson(1.8, n),
    0,
    10
)

# Recent percentage change in service usage
usage_change_pct = np.clip(
    rng.normal(-2, 18, n),
    -60,
    60
)

# Customer age
customer_age = np.clip(
    rng.normal(42, 14, n),
    18,
    85
).round()

# Average monthly service usage units
service_usage = np.clip(
    rng.normal(55, 18, n),
    5,
    120
)

# Discount rate as a percentage
discount_rate = np.clip(
    rng.normal(8, 6, n),
    0,
    30
)

# Satisfaction score on a 1–10 scale
satisfaction_score = np.clip(
    rng.normal(7, 1.8, n),
    1,
    10
)

# Contract type
contract_type = rng.choice(
    ["Month-to-month", "One year", "Two year"],
    size=n,
    p=[0.55, 0.25, 0.20]
)

# Automatic payment status
autopay = rng.choice(
    ["Yes", "No"],
    size=n,
    p=[0.62, 0.38]
)

print("Predictor variables generated.")

Predictor variables generated.


# 5. Generate the Regression Target: Annual Spending

We now create a continuous target called `annual_spending`.

The target is influenced by several customer characteristics plus random variation.

The exact values will depend on your Student ID seed.

This target is intentionally designed so that:

- some predictors are useful,
- some relationships are stronger than others,
- and the data are not perfectly predictable.

In [4]:
# Base annual spending relationship
annual_spending = (
    250
    + 9.5 * monthly_charge
    + 4.0 * service_usage
    + 2.2 * tenure_months
    - 18.0 * support_tickets
    + 7.5 * satisfaction_score
    - 6.0 * discount_rate
    + 0.7 * customer_age
)

# Add contract effects
annual_spending += np.where(contract_type == "One year", 120, 0)
annual_spending += np.where(contract_type == "Two year", 240, 0)

# Add autopay effect
annual_spending += np.where(autopay == "Yes", 75, 0)

# Add a nonlinear component that creates an opportunity for feature engineering
annual_spending += 0.08 * (service_usage ** 2)

# Add random noise so the outcome is not perfectly predictable
annual_spending += rng.normal(0, 220, n)

# Keep values positive and round to cents
annual_spending = np.clip(annual_spending, 100, None).round(2)

print("Annual spending target generated.")

Annual spending target generated.


# 6. Generate the Classification Target: Churn

We next create a binary churn target.

The model below creates an underlying churn probability for each customer.

Examples of the built-in relationships include:

- longer tenure tends to reduce churn risk,
- more support tickets tend to increase churn risk,
- falling service usage tends to increase churn risk,
- lower satisfaction tends to increase churn risk,
- month-to-month contracts tend to have higher churn risk,
- two-year contracts tend to have lower churn risk,
- customers without autopay tend to have higher churn risk.

These are assumptions built into the synthetic teaching dataset and should not be interpreted as causal findings about real customers.

In [5]:
# Linear churn score (logit)
logit = (
    -0.6
    - 0.030 * tenure_months
    + 0.012 * (monthly_charge - 75)
    + 0.34 * support_tickets
    - 0.020 * usage_change_pct
    - 0.32 * (satisfaction_score - 7)
    + 0.85 * (contract_type == "Month-to-month")
    - 0.55 * (contract_type == "Two year")
    + 0.40 * (autopay == "No")
)

# Convert the logit into a probability using the sigmoid function
prob_churn = 1 / (1 + np.exp(-logit))

# Generate the actual binary churn outcome
churn = rng.binomial(
    1,
    prob_churn
)

print("Churn target generated.")

Churn target generated.


# 7. Create the DataFrame

In [6]:
df = pd.DataFrame({
    "tenure_months": tenure_months,
    "monthly_charge": monthly_charge.round(2),
    "support_tickets": support_tickets,
    "usage_change_pct": usage_change_pct.round(2),
    "customer_age": customer_age.astype(int),
    "service_usage": service_usage.round(2),
    "discount_rate": discount_rate.round(2),
    "satisfaction_score": satisfaction_score.round(2),
    "contract_type": contract_type,
    "autopay": autopay,
    "annual_spending": annual_spending,
    "churn": churn
})

df.head()

,tenure_months,monthly_charge,support_tickets,usage_change_pct,customer_age,service_usage,discount_rate,satisfaction_score,contract_type,autopay,annual_spending,churn
0,65,90.56,2,-4.36,50,40.40,12.35,4.63,Month-to-month,Yes,1497.20,1
1,2,34.06,0,10.35,85,40.24,4.68,7.74,One year,No,1012.82,0
2,43,97.14,4,-16.70,56,56.96,7.53,4.64,One year,Yes,1921.49,0
3,66,66.17,2,-3.67,33,50.30,9.72,5.77,Two year,No,1781.26,0
4,25,52.54,1,11.40,18,48.51,3.66,6.30,Month-to-month,No,1187.49,0


# 8. Add Missing Values

Real business datasets often contain missing values.

To make the assignment more realistic, we will introduce a small number of missing values.

The missing-value locations are also generated using your Student ID-based random number generator.

Do not manually remove these rows before beginning the assignment.

You are expected to make and justify your own data-preparation decisions later.

In [7]:
df_assignment = df.copy()

# Add missing values to selected predictor columns
missing_plan = {
    "monthly_charge": 28,
    "usage_change_pct": 32,
    "satisfaction_score": 26,
    "contract_type": 24,
    "autopay": 20
}

for column, count in missing_plan.items():
    idx = rng.choice(
        df_assignment.index,
        size=count,
        replace=False
    )
    df_assignment.loc[idx, column] = np.nan

print("Missing values added.")

Missing values added.


# 9. Inspect Your Dataset

The following cells let you confirm that your individualized dataset was created successfully.

Do not begin modeling yet.

Save the dataset first.

In [8]:
print("Dataset shape:", df_assignment.shape)

print("\nMissing values:")
print(df_assignment.isna().sum())

print("\nChurn distribution:")
print(df_assignment["churn"].value_counts())

print("\nChurn proportions:")
print(df_assignment["churn"].value_counts(normalize=True).round(3))

Dataset shape: (2000, 12)

Missing values:
tenure_months          0
monthly_charge        28
support_tickets        0
usage_change_pct      32
customer_age           0
service_usage          0
discount_rate          0
satisfaction_score    26
contract_type         24
autopay               20
annual_spending        0
churn                  0
dtype: int64

Churn distribution:
churn
0    1180
1     820
Name: count, dtype: int64

Churn proportions:
churn
0    0.59
1    0.41
Name: proportion, dtype: float64


In [9]:
df_assignment.head(10)

,tenure_months,monthly_charge,support_tickets,usage_change_pct,customer_age,service_usage,discount_rate,satisfaction_score,contract_type,autopay,annual_spending,churn
0,65,90.56,2,-4.36,50,40.40,12.35,4.63,Month-to-month,Yes,1497.20,1
1,2,34.06,0,10.35,85,40.24,4.68,7.74,One year,No,1012.82,0
2,43,97.14,4,-16.70,56,56.96,7.53,4.64,One year,Yes,1921.49,0
3,66,66.17,2,-3.67,33,50.30,9.72,5.77,Two year,No,1781.26,0
4,25,52.54,1,11.40,18,48.51,3.66,6.30,Month-to-month,No,1187.49,0
5,64,90.85,1,21.63,45,97.98,10.49,9.76,One year,No,3051.47,0
6,22,93.80,2,19.95,41,65.07,5.87,6.47,One year,No,2314.46,0
7,45,85.25,0,-30.62,36,10.95,3.80,6.01,Month-to-month,Yes,1580.37,1
8,10,72.06,2,-16.75,43,79.41,9.49,8.79,One year,Yes,1909.54,0
9,57,75.37,5,-8.01,22,69.27,0.38,5.39,One year,Yes,2118.85,0


# 10. Variable Dictionary

| Variable | Type | Description |
|---|---|---|
| `tenure_months` | Numeric | Number of months the customer has been with the company |
| `monthly_charge` | Numeric | Monthly service charge |
| `support_tickets` | Numeric | Number of recent support requests |
| `usage_change_pct` | Numeric | Recent percentage change in service usage |
| `customer_age` | Numeric | Customer age in years |
| `service_usage` | Numeric | Average monthly service usage units |
| `discount_rate` | Numeric | Discount percentage applied to the customer |
| `satisfaction_score` | Numeric | Customer satisfaction score from 1 to 10 |
| `contract_type` | Categorical | Month-to-month, One year, or Two year |
| `autopay` | Categorical | Whether automatic payment is enabled |
| `annual_spending` | Continuous target | Annual customer spending; use for regression |
| `churn` | Binary target | 0 = Stay, 1 = Churn; use for classification |

# 11. Save Your Individual Dataset

Save your generated dataset as a CSV file.

The filename will include your Student ID.

Example:

`BAN8101_Assignment1_123456789.csv`

You will use this CSV file throughout Assignment 1.

In [10]:
output_filename = f"BAN8101_Assignment1_{STUDENT_ID}.csv"

df_assignment.to_csv(
    output_filename,
    index=False
)

print("Dataset saved as:")
print(output_filename)

Dataset saved as:
BAN8101_Assignment1_123456789.csv


# 12. Reproducibility Check

Run the notebook from the beginning again.

If you use the same Student ID, you should reproduce the same dataset.

This is one of the key ideas behind a random seed:

> The dataset is randomized, but the randomization is reproducible.

### Important Assignment Rule

If you need additional random numbers later in Assignment 1, use the same random number generator:

```python
rng = np.random.default_rng(STUDENT_ID)
```

and generate additional random values from `rng`.

Do not substitute arbitrary seeds such as:

```python
random_state=42
```

when the assignment specifically asks you to use your individualized seed.

Where a scikit-learn function requires a `random_state`, use:

```python
random_state=STUDENT_ID
```

unless the assignment instructions specify otherwise.

# 13. Before You Submit Assignment 1

Your analysis should use:

### Regression target
`annual_spending`

### Classification target
`churn`

You are responsible for:

- preparing the data,
- handling missing values,
- encoding categorical variables,
- choosing predictors,
- preventing data leakage,
- comparing models,
- interpreting metrics,
- performing cross-validation,
- applying Ridge and Lasso,
- engineering and testing new features,
- and explaining your modeling decisions.

The dataset generator gives you the data.

**It does not make the modeling decisions for you.**